In [2]:
import sys
import os

from pathlib import Path

from dotenv import load_dotenv
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np

load_dotenv()

sys.path.append("../../")
from src.lib.mlflow import MlflowHandler
from src.main import _ensure_java_home
mlflow_handler = MlflowHandler()

In [3]:
spark_app_name = os.getenv("SPARK_APP_NAME")
spark_master_url = os.getenv("SPARK_MASTER_URL")
postgres_url = os.getenv("POSTGRES_URL")
postgres_user = os.getenv("POSTGRES_USER")
postgres_password = os.getenv("POSTGRES_PASSWORD")
_ensure_java_home()

required = {
        "POSTGRES_URL": postgres_url,
        "POSTGRES_USER": postgres_user,
        "POSTGRES_PASSWORD": postgres_password,
    }

spark = (
        SparkSession.builder.appName(spark_app_name)
        .master(spark_master_url)
        .config("spark.jars.packages", "org.postgresql:postgresql:42.7.8")
        .config("spark.ui.showConsoleProgress", "false")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .getOrCreate()
)

25/11/05 19:30:12 WARN Utils: Your hostname, MacBook-Air-de-Yose.local resolves to a loopback address: 127.0.0.1; using 10.48.79.23 instead (on interface en0)
25/11/05 19:30:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/Users/yosesotomayor/Code/store/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/yosesotomayor/.ivy2/cache
The jars for the packages stored in: /Users/yosesotomayor/.ivy2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6d32911f-04d1-4cc6-a2c7-a0af619a3f5b;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.8 in central
	found org.checkerframework#checker-qual;3.49.5 in central
:: resolution report :: resolve 160ms :: artifacts dl 6ms
	:: modules in use:
	org.checkerframework#checker-qual;3.49.5 from central in [default]
	org.postgresql#postgresql;42.7.8 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	---------------------------

In [4]:
missing = [k for k, v in required.items() if not v]
if missing:
        raise RuntimeError(
            f"Missing required environment variables for JDBC connection: {', '.join(missing)}"
        )

jdbc_options = {
        "url": str(postgres_url),
        "dbtable": "articles",
        "user": str(postgres_user),
        "password": str(postgres_password),
        "driver": "org.postgresql.Driver",
    }

reader = spark.read.format("jdbc")
for k, v in jdbc_options.items():
        reader = reader.option(k, v)

df = reader.load()

df.show()

+----------+------------+--------------------+---------------+-----------------+------------------+-----------------------+-------------------------+-----------------+-----------------+-------------------------+---------------------------+--------------------------+----------------------------+-------------+--------------------+----------+--------------------+--------------+----------------+----------+--------------------+----------------+------------------+--------------------+
|article_id|product_code|           prod_name|product_type_no|product_type_name|product_group_name|graphical_appearance_no|graphical_appearance_name|colour_group_code|colour_group_name|perceived_colour_value_id|perceived_colour_value_name|perceived_colour_master_id|perceived_colour_master_name|department_no|     department_name|index_code|          index_name|index_group_no|index_group_name|section_no|        section_name|garment_group_no|garment_group_name|         detail_desc|
+----------+------------+-------

In [5]:
jdbc_options_2 = {
        "url": str(postgres_url),
        "dbtable": "customers",
        "user": str(postgres_user),
        "password": str(postgres_password),
        "driver": "org.postgresql.Driver",
    }

reader_2 = spark.read.format("jdbc")
for k, v in jdbc_options_2.items():
        reader_2 = reader_2.option(k, v)

df_customers = reader_2.load()

df_customers.show()

+--------------------+----+------+------------------+----------------------+---+--------------------+----+-----+-------------+----------------+----------+
|         customer_id|  fn|active|club_member_status|fashion_news_frequency|age|         postal_code|name|email|password_hash|is_authenticated|created_at|
+--------------------+----+------+------------------+----------------------+---+--------------------+----+-----+-------------+----------------+----------+
|f55a003032a8c8f81...|NULL|  NULL|        PRE-CREATE|                  NONE| 36|86ab807568b3dd0a9...|NULL| NULL|         NULL|           false|      NULL|
|f55a0c2cc5df55162...|NULL|  NULL|            ACTIVE|                  NONE| 20|f084a0d336e36861b...|NULL| NULL|         NULL|           false|      NULL|
|f55a13eeb44c04f22...|NULL|  NULL|            ACTIVE|                  NONE| 27|a29ed3dd3b856183d...|NULL| NULL|         NULL|           false|      NULL|
|f55a245f40d0b8690...|NULL|  NULL|            ACTIVE|                 

In [6]:
df_customers_pandas = df_customers.toPandas()

In [7]:
df_pandas = df.toPandas()

In [8]:
pd.set_option('display.max_columns', None)
df_pandas

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,709415003,709415,IKE romper,267,Jumpsuit/Playsuit,Garment Full body,1010006,Dot,8,Dark Grey,2,Medium Dusty,12,Grey,6564,Newborn,G,Baby Sizes 50-98,4,Baby/Children,44,Baby Essentials & Complements,1005,Jersey Fancy,"All-in-one suit in soft, printed organic cotto..."
1,709415004,709415,IKE romper,267,Jumpsuit/Playsuit,Garment Full body,1010001,All over pattern,51,Light Pink,1,Dusty Light,4,Pink,6564,Newborn,G,Baby Sizes 50-98,4,Baby/Children,44,Baby Essentials & Complements,1005,Jersey Fancy,"All-in-one suit in soft, printed organic cotto..."
2,709418001,709418,DIV Anni oversize hood,252,Sweater,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1652,Divided+,D,Divided,2,Divided,50,Divided Projects,1001,Unknown,Oversized top in sweatshirt fabric with a line...
3,709419001,709419,KAROLIN dress,265,Dress,Garment Full body,1010001,All over pattern,10,White,3,Light,9,White,6564,Newborn,G,Baby Sizes 50-98,4,Baby/Children,44,Baby Essentials & Complements,1005,Jersey Fancy,"Short-sleeved dress in soft, patterned organic..."
4,709433002,709433,Ella Mae heel,90,Pumps,Shoes,1010016,Solid,9,Black,4,Dark,5,Black,3528,Heels,C,Ladies Accessories,1,Ladieswear,64,Womens Shoes,1020,Shoes,Imitation suede sandals with narrow ankle stra...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105537,709412001,709412,BILLY braces set,270,Garment Set,Garment Full body,1010001,All over pattern,23,Dark Yellow,2,Medium Dusty,13,Brown,6564,Newborn,G,Baby Sizes 50-98,4,Baby/Children,44,Baby Essentials & Complements,1005,Jersey Fancy,Set with a long-sleeved bodysuit and pair of d...
105538,709414001,709414,Flirty Ronny necklace,77,Necklace,Accessories,1010016,Solid,5,Gold,5,Bright,15,Metal,4344,Jewellery,C,Ladies Accessories,1,Ladieswear,66,Womens Small accessories,1019,Accessories,"Thin metal chain necklaces with round, coin-sh..."
105539,709414002,709414,Flirty Ronny necklace,77,Necklace,Accessories,1010016,Solid,3,Silver,3,Light,15,Metal,4344,Jewellery,C,Ladies Accessories,1,Ladieswear,66,Womens Small accessories,1019,Accessories,"Thin metal chain necklaces with round, coin-sh..."
105540,709415001,709415,IKE romper,267,Jumpsuit/Playsuit,Garment Full body,1010001,All over pattern,6,Light Grey,1,Dusty Light,12,Grey,6564,Newborn,G,Baby Sizes 50-98,4,Baby/Children,44,Baby Essentials & Complements,1005,Jersey Fancy,"All-in-one suit in soft, printed organic cotto..."


In [9]:
df_customers_pandas

,customer_id,fn,active,club_member_status,fashion_news_frequency,age,postal_code,name,email,password_hash,is_authenticated,created_at
0,f55a003032a8c8f810a465f64bf9e36bafd896d29524bc...,NaN,NaN,PRE-CREATE,NONE,36.0,86ab807568b3dd0a9cea497ac473d7910a839b411e774d...,None,None,None,False,NaT
1,f55a0c2cc5df55162161cce2f134980b3d0bdf9af5982d...,NaN,NaN,ACTIVE,NONE,20.0,f084a0d336e36861b4b1b7a7d4ad39df697b79683c01f4...,None,None,None,False,NaT
2,f55a13eeb44c04f22783670cadab66c56d38750a4beffc...,NaN,NaN,ACTIVE,NONE,27.0,a29ed3dd3b856183db3a6dc8fc6a92a2bc1b1a627874d0...,None,None,None,False,NaT
3,f55a245f40d0b86905b4f615ca25e1c34f1bc5d9fabc39...,NaN,NaN,ACTIVE,NONE,53.0,5ba37477d8e5c3af86682c81f0018ac6bb85966a2fc757...,None,None,None,False,NaT
4,f55a2a5115aae273613a9a730ec26f141d79d9272409a5...,NaN,NaN,ACTIVE,NONE,25.0,08ee43710825ec81cc002e340689feb4f7ee320615a4b0...,None,None,None,False,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...
1371985,f559809eebef0671d5b5be737fabd134c913a29df98acb...,1.0,1.0,ACTIVE,Regularly,27.0,2c29ae653a9282cce4151bd87643c907644e09541abc28...,None,None,None,False,NaT
1371986,f5598e6e593f03740cbef42776cbf2dc20f379e8bebdb1...,1.0,1.0,ACTIVE,Regularly,24.0,2c29ae653a9282cce4151bd87643c907644e09541abc28...,None,None,None,False,NaT
1371987,f559ba6cd345bbe6f44547296bc03c906024441a6e5b33...,NaN,NaN,PRE-CREATE,NONE,NaN,502a058326ac4e090a3d3e1ffb53388f7c3713ad99bbe6...,None,None,None,False,NaT
1371988,f559beab8ebb0ec81ae812f6519cde4274250cf121f20e...,1.0,1.0,ACTIVE,Regularly,49.0,e0f349c59150ee8ab30f647c9b3ea38a1ef04f9494e25c...,None,None,None,False,NaT


In [10]:
df_transactions_pandas = pd.read_csv("/Users/yosesotomayor/Desktop/data_store/transactions_train.csv")

In [11]:
df_pandas['product_type_name'].nunique()

131

In [12]:
df_transactions_pandas = df_transactions_pandas[['customer_id', 'article_id']]

In [13]:
df_transactions_pandas.set_index(['customer_id'], inplace=True)

In [14]:
inter_ = df_transactions_pandas.index.intersection(df_customers_pandas['customer_id'])

In [15]:
df_transactions_pandas.groupby('customer_id').count()

,article_id
customer_id,
00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657,21
0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa,86
000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,18
00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2c5feb1ca5dff07c43e,2
00006413d8573cd20ed7128e53b7b13819fe5cfc2d801fe7fc0f26dd8d65a85a,13
...,...
ffffbbf78b6eaac697a8a5dfbfd2bfa8113ee5b403e4747568cac33e8c541831,51
ffffcd5046a6143d29a04fb8c424ce494a76e5cdf4fab53481233731b5c4f8b7,84
ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1778d0116cffd259264,45


In [16]:
prueba = df_transactions_pandas.loc["00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657"]

In [17]:
df_pandas.index = df_pandas.pop('article_id')
df_pandas.loc[prueba['article_id'].values]

,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
article_id,,,,,,,,,,,,,,,,,,,,,,,,
625548001,625548,BB Chris puff jkt TP,262,Jacket,Garment Upper body,1010016,Solid,73,Dark Blue,4,Dark,2,Blue,8852,Young Boy Outdoor,I,Children Sizes 134-170,4,Baby/Children,45,Kids Outerwear,1007,Outdoor,"Padded jacket with a detachable hood, stand-up..."
176209023,176209,Mr Harrington w/hood,308,Hoodie,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,5283,Jacket Street,F,Menswear,3,Menswear,31,Mens Outerwear,1007,Outdoor,"Short, padded jacket with a jersey-lined hood ..."
627759010,627759,FLORA parka,262,Jacket,Garment Upper body,1010016,Solid,73,Dark Blue,4,Dark,2,Blue,7812,Kids Girl Outdoor,H,Children Sizes 92-140,4,Baby/Children,45,Kids Outerwear,1007,Outdoor,"Padded parka in woven fabric with a soft, brus..."
697138006,697138,Sophie jumpsuit,267,Jumpsuit/Playsuit,Garment Full body,1010001,All over pattern,51,Light Pink,1,Dusty Light,4,Pink,7616,Kids Girl Jersey Fancy,H,Children Sizes 92-140,4,Baby/Children,76,Kids Girl,1005,Jersey Fancy,Playsuit in cotton jersey with butterfly sleev...
568601006,568601,Mariette Blazer,264,Blazer,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1212,Suit,A,Ladieswear,1,Ladieswear,11,Womens Tailoring,1008,Dressed,Fitted jacket in woven fabric with notch lapel...
568601006,568601,Mariette Blazer,264,Blazer,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1212,Suit,A,Ladieswear,1,Ladieswear,11,Womens Tailoring,1008,Dressed,Fitted jacket in woven fabric with notch lapel...
607642008,607642,The Firm (1),259,Shirt,Garment Upper body,1010017,Stripe,9,Black,4,Dark,5,Black,1515,Blouse,A,Ladieswear,1,Ladieswear,11,Womens Tailoring,1010,Blouses,Top in a crêpe weave with a V-shaped opening a...
745232001,745232,Skirt Mini Stretch Edie,275,Skirt,Garment Lower body,1010023,Denim,9,Black,4,Dark,5,Black,1773,Denim Other Garments,D,Divided,2,Divided,57,Ladies Denim,1016,Trousers Denim,"Short 5-pocket skirt in washed, stretch denim ..."
656719005,656719,Serpente HW slim trouser,272,Trousers,Garment Lower body,1010016,Solid,9,Black,4,Dark,5,Black,1722,Trouser,A,Ladieswear,1,Ladieswear,15,Womens Everyday Collection,1009,Trousers,Tailored trousers in a stretch weave with two ...


In [18]:
df_pandas.info()

<class 'pandas.core.frame.DataFrame'>
Index: 105542 entries, 709415003 to 709415002
Data columns (total 24 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   product_code                  105542 non-null  int32 
 1   prod_name                     105542 non-null  object
 2   product_type_no               105542 non-null  int32 
 3   product_type_name             105542 non-null  object
 4   product_group_name            105542 non-null  object
 5   graphical_appearance_no       105542 non-null  int32 
 6   graphical_appearance_name     105542 non-null  object
 7   colour_group_code             105542 non-null  int32 
 8   colour_group_name             105542 non-null  object
 9   perceived_colour_value_id     105542 non-null  int32 
 10  perceived_colour_value_name   105542 non-null  object
 11  perceived_colour_master_id    105542 non-null  int32 
 12  perceived_colour_master_name  105542 non-null  objec

In [19]:
df_transactions_pandas.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31788324 entries, 000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318 to fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20e02ce5d1e58a8f700b
Data columns (total 1 columns):
 #   Column      Dtype
---  ------      -----
 0   article_id  int64
dtypes: int64(1)
memory usage: 517.3+ MB


In [20]:
from implicit.als import AlternatingLeastSquares
from scipy.sparse import coo_matrix

customer_map = {cid: i for i, cid in enumerate(df_transactions_pandas.index.unique())}
article_map = {aid: i for i, aid in enumerate(df_pandas.index)}

rows = df_transactions_pandas.index.map(customer_map.get)
cols = df_transactions_pandas['article_id'].map(article_map.get)
values = [1] * len(df_transactions_pandas)

interaction_matrix = coo_matrix((values, (rows, cols)))
interaction_matrix_csr = interaction_matrix.tocsr()

model = AlternatingLeastSquares(factors=50, regularization=0.1, iterations=15)
model.fit(interaction_matrix)

/Users/yosesotomayor/Code/store/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/yosesotomayor/Code/store/.venv/lib/python3.11/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.9744288921356201 seconds
  warnings.warn(
100%|██████████| 15/15 [01:39<00:00,  6.63s/it]


In [21]:
inv_article_map = {v: k for k, v in article_map.items()}

random_customer_id = df_transactions_pandas.sample(1).index[0]
customer_idx = customer_map[random_customer_id]

recommended = model.recommend(
    customer_idx,
    interaction_matrix_csr[customer_idx],
    N=10,
    recalculate_user=True,              
    filter_already_liked_items=True     
)

if isinstance(recommended, list):
    item_indices = [it for it, _ in recommended]
    scores = [sc for _, sc in recommended]
elif isinstance(recommended, tuple) and len(recommended) == 2:
    item_indices, scores = recommended
else:
    raise TypeError(f"Formato inesperado de recommended: {type(recommended)}")

recommended_article_ids = [inv_article_map[int(i)] for i in item_indices]

out = (
    pd.DataFrame({
        "article_id": recommended_article_ids,
        "score": scores
    })
    .merge(
        df_pandas[["prod_name","product_type_name","garment_group_name","colour_group_name"]],
        on="article_id",
        how="left"
    )
    .sort_values("score", ascending=False, kind="mergesort")
    .reset_index(drop=True)
)

print(f"Cliente {random_customer_id} → historial-de-compras")
display(df_pandas.loc[df_transactions_pandas.loc[random_customer_id]['article_id'].values, ["prod_name","product_type_name","garment_group_name","colour_group_name"]])

print(f"Cliente {random_customer_id} → top-{len(out)} recomendaciones")
display(out.drop(columns=['score']))

Cliente 7b8625c9a7695ead04224c95816462728053bde1be4d82986a7ed567bb65dea0 → historial-de-compras


,prod_name,product_type_name,garment_group_name,colour_group_name
article_id,,,,
578470001,Devon coat,Coat,Outdoor,Black
639448005,Case tote,Bag,Accessories,Dark Turquoise
711440002,Emerald,Sweater,Jersey Fancy,Red
711440001,Emerald (1),Sweater,Jersey Fancy,Black
669882001,Thelma tie top,Top,Jersey Fancy,Black
578476001,Vichy 5pkt slim trouser,Trousers,Trousers,Black
573716012,Kanta slacks RW,Trousers,Trousers,Black
636892002,Nonius,Jacket,Outdoor,Black
634426005,Samantha,Blouse,Blouses,Dark Blue


Cliente 7b8625c9a7695ead04224c95816462728053bde1be4d82986a7ed567bb65dea0 → top-10 recomendaciones


,article_id,prod_name,product_type_name,garment_group_name,colour_group_name
0,568601006,Mariette Blazer,Blazer,Dressed,Black
1,678942001,Harrison short sleeve top CN,Top,Knitwear,Black
2,841383003,Vanessa 2-pack,Vest top,Jersey Basic,White
3,562245046,Luna skinny RW,Trousers,Trousers,Black
4,568597006,Hayes slim trouser,Trousers,Trousers,Black
5,783346001,Primo slacks,Trousers,Trousers,Black
6,723469001,Kelly 2pk Melbourne push ct,Bra,"Under-, Nightwear",Black
7,787946002,Moa 2 pack tank,Vest top,Jersey Basic,Black
8,688537011,Simple as that Cheeky Tanga,Swimwear bottom,Swimwear,Dark Green
9,590928001,New Girl Push Top,Bikini top,Swimwear,Black


In [22]:
def dar_recomendaciones(customer_id):
    customer_idx = customer_map.get(customer_id)
    if customer_idx is None:
        raise ValueError(f"Cliente ID {customer_id} no encontrado en el mapa de clientes.")
    
    recommended = model.recommend(
        customer_idx,
        interaction_matrix_csr[customer_idx],
        N=10,
        recalculate_user=True,              
        filter_already_liked_items=True     
    )

    if isinstance(recommended, list):
        item_indices = [it for it, _ in recommended]
        scores = [sc for _, sc in recommended]
    elif isinstance(recommended, tuple) and len(recommended) == 2:
        item_indices, scores = recommended
    else:
        raise TypeError(f"Formato inesperado de recommended: {type(recommended)}")

    recommended_article_ids = [inv_article_map[int(i)] for i in item_indices]

    out = (
        pd.DataFrame({
            "article_id": recommended_article_ids,
            "score": scores
        })
        .merge(
            df_pandas[["prod_name","product_type_name","garment_group_name","colour_group_name"]],
            on="article_id",
            how="left"
        )
        .sort_values("score", ascending=False, kind="mergesort")
        .reset_index(drop=True)
    )
    return out

customer_id_ejemplo = df_transactions_pandas.sample(1).index[0]
recomendaciones = dar_recomendaciones(customer_id_ejemplo)
print(f"Recomendaciones para el cliente {customer_id_ejemplo}:")
display(recomendaciones.drop(columns=['score']))

Recomendaciones para el cliente eea5808cf53fffd09fc7aeb48122bf697df3d2609b8d8d255368e6dfe1b1c234:


,article_id,prod_name,product_type_name,garment_group_name,colour_group_name
0,573937001,ED Madison Skinny HW,Trousers,Trousers,Black
1,294008002,HM+ Cora tee,Costumes,Jersey Fancy,Black
2,652924004,&DENIM Jeggings HW,Trousers,Trousers Denim,Black
3,751551001,HM+ Track dress,Dress,Jersey Fancy,Black
4,368979001,ED Long leggings,Leggings/Tights,Jersey Fancy,Black
5,779551002,DIV Tess tee,Top,Jersey Basic,Black
6,779554002,DIV Nicky tank top,Top,Jersey Basic,Black
7,750330002,&DENIM Skinny HW Ancle Vanessa,Trousers,Trousers Denim,Blue
8,652924010,&DENIM Jeggings HW,Trousers,Trousers Denim,Blue
9,624257001,&DENIM+ Skinny shaping RW,Trousers,Trousers Denim,Black


In [36]:
import mlflow, pickle, json, tempfile
import pandas as pd

# Wrapper mínimo para servir recomendaciones
class ALSWrapper(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        import pickle, json
        with open(context.artifacts["als_model"], "rb") as f:
            self.model = pickle.load(f)
        with open(context.artifacts["user_map"], "r") as f:
            self.user_map = json.load(f)
        with open(context.artifacts["item_map"], "r") as f:
            self.item_map = json.load(f)
        self.rev_item_map = {int(v): k for k, v in self.item_map.items()}

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        out = []
        for _, r in model_input.iterrows():
            uid = r["user_id"]
            N = int(r["N"]) if "N" in r and pd.notna(r["N"]) else 10
            uidx = int(self.user_map[str(uid)])
            # Si tienes matriz CSR, úsala: user_items = interaction_matrix_csr[uidx]
            recs = self.model.recommend(uidx, user_items=None, N=N)
            for iidx, score in recs:
                out.append({"user_id": uid, "item_id": self.rev_item_map.get(int(iidx), int(iidx)), "score": float(score)})
        return pd.DataFrame(out)

with mlflow.start_run(run_name="als recomendaciones") as run:
    run_id = run.info.run_id
    print("Run ID:", run_id)

    with tempfile.TemporaryDirectory() as tmp:
        pickle.dump(model, open(f"{tmp}/model.pkl", "wb"))
        json.dump(customer_map, open(f"{tmp}/user_map.json", "w"))
        json.dump(article_map, open(f"{tmp}/item_map.json", "w"))

        mlflow.pyfunc.log_model(
            artifact_path="model",
            python_model=ALSWrapper(),
            artifacts={
                "als_model": f"{tmp}/model.pkl",
                "user_map": f"{tmp}/user_map.json",
                "item_map": f"{tmp}/item_map.json",
            },
            pip_requirements=["mlflow>=3.0.0","implicit>=0.7","pandas","numpy","scipy"],
        )

model_uri = f"runs:/{run_id}/model"
print("Model URI:", model_uri)

Run ID: ab5440ef3df84f7bbc76d8bb5843233c


2025/11/05 19:46:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/05 19:46:34 WARNING mlflow.pyfunc: Failed to infer model signature: Type hint <input: <class 'pandas.core.frame.DataFrame'>, output: <class 'pandas.core.frame.DataFrame'>> cannot be used to infer model signature and input example is not provided, model signature cannot be inferred.
2025/11/05 19:46:35 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/11/05 19:46:35 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run als recomendaciones at: http://100.64.101.26:5000/#/experiments/1/runs/ab5440ef3df84f7bbc76d8bb5843233c
🧪 View experiment at: http://100.64.101.26:5000/#/experiments/1


S3UploadFailedError: Failed to upload /var/folders/gw/6t_48cb11cq3wt9bjf_ns5p40000gn/T/tmp603lhj8d/model/python_env.yaml to mlflow/1/models/m-028189e5ef9f4b2ab7f0bcf626228ec9/artifacts/python_env.yaml: An error occurred (AccessDenied) when calling the PutObject operation: Access Denied

25/11/05 19:52:28 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 256553 ms exceeds timeout 120000 ms
25/11/05 19:52:28 WARN SparkContext: Killing executors is not supported by current scheduler.
25/11/05 19:52:35 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$